# 第 13 章:对齐 II —— RLAIF:PPO / GRPO / CISPO 统一框架

第 12 章我们学了 DPO —— 一种不需要在线采样的离线对齐方法。但 DPO 依赖于预先收集的偏好数据,无法探索新的策略空间。

本章进入**在线强化学习对齐(RLAIF)**:模型自己生成回复,获得奖励,再更新自身。这是 ChatGPT、DeepSeek 等顶级模型对齐阶段的核心技术。

minimind 完整实现了三种 RLAIF 算法:**PPO**、**GRPO** 和 **CISPO**。本章的核心洞察是:它们其实是**同一个框架的三种实例**。

## 13.1 统一 PO 视角(本章灵魂)

在深入具体算法之前,先建立一个统一视角。所有 Policy Optimization(策略优化)算法都可以写成:

$$\mathcal{L} = \underbrace{\text{policy\_term}(r_t)}_{\text{策略项}} \cdot \underbrace{A_t}_{\text{优势}} - \underbrace{\beta \cdot \text{KL}(\pi \| \pi_{\text{ref}})}_{\text{KL 正则}}$$

三个可替换的组件:

| 算法 | policy_term | advantage | KL 形式 |
|---|---|---|---|
| **DPO** | 隐式(通过偏好对) | chosen−rejected | β 固定 |
| **PPO** | `min(r·A, clip(r)·A)` | GAE 时序差分 | 惩罚项 |
| **GRPO** | `min(r·A, clip(r)·A)` | 组归一化 | per-token KL |
| **CISPO** | `clamp(r, max=ε_h)·A` | 组归一化 | per-token KL |

> **关键区别**:PPO 用 **GAE**(需要 Critic 模型)算优势;GRPO/CISPO 用**组归一化**(不需要 Critic)算优势。CISPO 与 GRPO 的区别只在 policy_term 的裁剪方式。

## 13.2 Rollout Engine:RL 的「采样器」

RL 训练需要反复「生成回复 → 计算奖励 → 更新策略」。这个「生成」步骤由 **rollout engine** 完成。

读 `rollout_engine.py`:

```
RolloutEngine (抽象接口)
├── TorchRolloutEngine   → 直接用 model.generate
└── SGLangRolloutEngine  → HTTP 调用 sglang 服务
```

核心函数 `compute_per_token_logps`(24-36 行):用 `logits_to_keep` 只计算 completion 部分的 log-prob,跳过 prompt 部分 —— 大幅节省计算。

### 逐 token log-prob 的计算

```python
# rollout_engine.py:24-36 的核心逻辑
def compute_per_token_logps(model, input_ids, logits_to_keep):
    # logits_to_keep: 只保留最后 N 个位置的 logits(completion 部分)
    logits = model(input_ids, logits_to_keep=logits_to_keep).logits
    # Shape: (batch, completion_len, vocab_size)
    log_probs = F.log_softmax(logits, dim=-1)
    # 收集实际生成 token 的 log-prob
    # Shape: (batch, completion_len)
    per_token_logps = log_probs.gather(-1, target.unsqueeze(-1)).squeeze(-1)
    return per_token_logps
```

> **为什么逐 token?** 序列级 log-prob = 所有 token log-prob 之和。但 RL 需要逐 token 的信号来分配奖励 —— 这就是「credit assignment」问题。

## 13.3 PPO:4 个模型的庞然大物

PPO(Proximal Policy Optimization)是 OpenAI 用于 ChatGPT 对齐的经典算法。minimind 的实现(`train_ppo.py`)需要**4 个模型**:

| 模型 | 角色 | 是否训练 |
|---|---|---|
| **Actor** | 策略 π(生成回复) | ✅ 训练 |
| **Critic** | 价值函数 V(s)(估计状态价值) | ✅ 训练 |
| **Ref** | 冻结的 SFT 副本(KL 参考) | ❌ 冻结 |
| **Reward** | 奖励模型(打分) | ❌ 冻结 |

### Critic 模型

Critic 和 Actor 结构几乎一样,只是把 `lm_head`(输出词表 logits)换成 `value_head`(输出标量 V(s)):

```python
# train_ppo.py:36-49 (@67f114a)
class CriticModel(MiniMindForCausalLM):
    def __init__(self, config):
        super().__init__(config)
        self.value_head = nn.Linear(config.hidden_size, 1, bias=False)
        # 替换 lm_head

    def forward(self, input_ids):
        hidden = self.model(input_ids)
        # Shape: (batch, seq_len, hidden_size)
        value = self.value_head(hidden)
        # Shape: (batch, seq_len, 1) → 标量价值
        return value.squeeze(-1)
```

### 奖励函数

`calculate_rewards`(52-76 行)组合多种信号:

- **长度奖励**:鼓励适当长度(太短=敷衍,太长=啰嗦)
- **`<think>` 奖励**:鼓励思考过程(有 `<think>` 标签 → 加分)
- **重复惩罚**:检测重复 n-gram → 减分
- **RM 分数**:奖励模型的连续打分,clamp 到 [-3, 3]

```python
reward = length_bonus + think_bonus + rep_penalty + rm_score
reward = max(-3.0, min(3.0, reward))  # clamp
```

### GAE:广义优势估计

**优势(Advantage)** $A_t$ 衡量「在状态 $s_t$ 采取动作 $a_t$ 比平均好多少」:

$$A_t = Q(s_t, a_t) - V(s_t)$$

但直接算 $Q$ 需要完整轨迹。**GAE**(Generalized Advantage Estimation)用 Critic 的 $V$ 来估计:

$$\hat{A}_t = \sum_{l=0}^{\infty} (\gamma \lambda)^l \delta_{t+l}$$

其中 $\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$ 是 TD 误差。

```python
# train_ppo.py:139-150 (@67f114a, 简化版)
def compute_gae(rewards, values, gamma=0.99, lam=0.95):
    advantages = []
    gae = 0
    for t in reversed(range(len(rewards))):
        delta = rewards[t] + gamma * values[t+1] - values[t]
        gae = delta + gamma * lam * gae
        advantages.insert(0, gae)
    return advantages  # 从后往前累加
```

### PPO Clip Loss

PPO 的核心是**裁剪的重要性采样比率**:

$$r_t = \frac{\pi_\theta(a_t|s_t)}{\pi_{\text{old}}(a_t|s_t)}$$

$$\mathcal{L}_{\text{policy}} = -\min(r_t \cdot A_t,\ \text{clip}(r_t, 1-\epsilon, 1+\epsilon) \cdot A_t)$$

- $r_t > 1$:新策略比旧策略更倾向这个动作
- $\epsilon = 0.2$:允许 20% 的比率偏移,超过就裁剪
- **裁剪的目的**:防止策略更新过大导致崩溃

```python
# train_ppo.py:204-216 (@67f114a)
ratio = torch.exp(new_logps - old_logps)
surr1 = ratio * advantages
surr2 = torch.clamp(ratio, 1 - eps, 1 + eps) * advantages
policy_loss = -torch.min(surr1, surr2).mean()
```

总 loss = policy_loss + vf_coef × value_loss + kl_coef × KL_penalty

### 实现细节:三处行为变更(@67f114a)

上面的公式是 PPO 的数学骨架,但 master 版本的工程实现做了三处关键调整,理解它们能避免复现时踩坑:

**① `mb_resp_logp` 移入 autocast 上下文(数值稳定性修正)**

PPO 的比率 `ratio = exp(mb_resp_logp − old_resp_logp)` 要求新旧 log-prob 在同一数值精度下计算。早期版本中,`mb_resp_logp = F.log_softmax(...)` 写在 `with autocast_ctx:` 块**之外**,意味着它直接对 fp16/bf16 的 logits 做 log_softmax —— 这会引入额外的舍入误差,让首轮 ratio 偏离 1。

```python
# train_ppo.py:~173 (@67f114a) — 修正后
with autocast_ctx:
    res = actor_unwrapped(input_ids=..., attention_mask=...)
    aux_loss = res.aux_loss if lm_config.use_moe else ...
    # ✅ log_softmax 移入 autocast,框架在正确精度下计算
    mb_resp_logp = F.log_softmax(res.logits[:, :-1], dim=-1)...
```

> **教学要点**:混合精度下,任何涉及 softmax / log_softmax 的运算都应在 autocast 上下文内完成,让 PyTorch 自动选择合适的内部精度。这是一个**真实的 bugfix**(不是 cosmetic 改动),它直接影响 PPO 的 `ratio ≈ 1` 假设在首轮的成立程度。

**② `ppo_train_epoch` 移除 `use_sglang` 形参**

```python
# 旧版:  ppo_train_epoch(..., use_sglang=False)
# 新版:  ppo_train_epoch(...)   ← 无 use_sglang
```

sglang 路由逻辑已**下沉到 `rollout_engine` 内部**,`ppo_train_epoch` 不再需要感知采样后端类型。调用处也从 `use_sglang=(args.rollout_engine=="sglang")` 简化为直接传递 `rollout_engine` 对象。这是**关注点分离**的体现:训练循环只关心「采样 → 算优势 → 更新策略」,不关心采样用哪个后端。

**③ `--debug_log_ratio` 诊断工具(可选,默认关闭)**

新增 CLI 参数 `--debug_log_ratio`,在**首轮首个 minibatch** 打印 ratio 的实际分布:

```python
# train_ppo.py:~183 (@67f114a)
if args.debug_log_ratio and ppo_epoch == 0 and i == 0:
    Logger(f"[DBG log_ratio] max|lr|=... ratio_max=... ratio_min=...")
```

> **为什么需要它?** PPO 的理论前提是首轮 `ratio ≈ 1`(新旧策略几乎相同)。如果训练初期就出现大的 ratio 偏差,说明 rollout 与训练的数值路径不一致(如 dropout、精度、stop-gradient 等)。这个开关帮你快速定位这类问题。**默认关闭**,不影响正常训练。

> **其他次要变更**(@67f114a):移除了未使用的 `from transformers import AutoTokenizer`;checkpoint 后缀从内联 `moe_suffix` 改用统一的 `_model_suffix(lm_config)`(与第 8 章预训练一致),无行为影响。

## 13.4 GRPO:去掉 Critic

GRPO(Group Relative Policy Optimization)的核心创新:**不需要 Critic 模型**。

思路:对每个 prompt 生成 **N 个回复**(minimind 中 N=6),用组内统计代替 Critic:

$$A_i = \frac{r_i - \mu_{\text{group}}}{\sigma_{\text{group}} + \epsilon}$$

```python
# train_grpo.py:121-124
rewards = torch.tensor([r1, r2, r3, r4, r5, r6])  # 6 个回复的奖励
advantages = (rewards - rewards.mean()) / (rewards.std() + 1e-4)
# 归一化:高于均值的正优势,低于均值的负优势
```

> **为什么有效?** Critic 的作用是提供 baseline(「平均能拿多少分」)。如果对同一 prompt 生成多个回复,组均值就是一个天然的 baseline —— 不需要专门训练 Critic。

### GRPO Loss

```python
# train_grpo.py:135-142 (grpo 模式)
ratio = torch.exp(new_logps - old_logps)
surr1 = ratio * advantages
surr2 = torch.clamp(ratio, 1 - eps, 1 + eps) * advantages
grpo_loss = -torch.min(surr1, surr2).mean()

# 加 per-token KL 正则
kl = torch.exp(old_logps) * (old_logps - new_logps) - 1  # exp(kl)-kl-1
loss = grpo_loss + beta * kl.mean()
```

KL 使用 `exp(kl) - kl - 1` 形式(比标准 KL 更稳定,对大偏差惩罚更强)。

## 13.5 CISPO:minimind 的默认选择

CISPO 是 minimind 的默认 RL 算法(`loss_type='cispo'`)。与 GRPO 的区别**仅在 policy_term**:

| | GRPO | CISPO |
|---|---|---|
| policy_term | `min(r·A, clip(r)·A)` | `clamp(r, max=ε_h)·A` |
| 正向更新 | 被 clip 限制 | 不限制(只要 r < ε_h) |
| 负向更新 | 被 clip 限制 | 正常执行 |

```python
# train_grpo.py:135-142 (cispo 模式)
ratio = torch.exp(new_logps - old_logps)
# CISPO: 只裁剪上界,不裁剪下界
clipped_ratio = torch.clamp(ratio, max=epsilon_high)  # ε_high = 5.0
cispo_loss = -(clipped_ratio * advantages * new_logps).mean()
```

> **为什么更稳定?** CISPO 允许大的正向更新(当优势为正且比率适中时),只限制极端的正向偏离(比率 > ε_high)。这让模型更快学习好的行为,同时防止策略崩溃。

## 13.6 训练配置对比

| 配置 | PPO | GRPO | CISPO |
|---|---|---|---|
| 模型数 | 4(Actor+Critic+Ref+RM) | 3(Policy+Ref+RM) | 3(Policy+Ref+RM) |
| 每提示生成数 | 1 | 6 | 6 |
| 优势计算 | GAE(需要 Critic) | 组归一化 | 组归一化 |
| lr | 3e-7 | 3e-7 | 3e-7 |
| clip ε | 0.2 | 0.2 | ε_high=5.0 |
| KL 系数 β | 0.02 | 0.1 | 0.1 |

> **显存**:PPO 需要 4 个模型 → 显存压力最大;GRPO/CISPO 只需 3 个 → 更轻量。

## 13.7 奖励稀疏问题

minimind 是 0.1B 模型,无法解决 MATH500 等难题。如果用 binary rule reward(答对=1,答错=0):

- 大部分回复 → reward = 0
- advantage = (0 - 0) / σ = 0
- **学不到任何东西**

解决方案:minimind 使用 **dense reward**(奖励模型的连续分数),即使答错也有部分分数。这保证了梯度信号不会消失。

> 这是小模型 RL 的核心挑战:在能力不足时,如何提供有效的学习信号?minimind 的答案是「用 RM 的连续打分替代 binary 规则验证」。

&nbsp;

---

## Summary and takeaways

- 所有 PO 算法 = `policy_term · advantage − KL`,三者可替换
- **PPO**:4 模型,GAE 优势,clip 裁剪 —— 经典但重
- **GRPO**:3 模型,组归一化优势 —— 去 Critic,更轻量
- **CISPO**:clamp 上界裁剪 —— 允许大正向更新,更稳定(minimind 默认)
- 奖励稀疏是小模型 RL 的核心挑战,dense RM score 是解法

> **核心认知**:RL 对齐不是魔法,它只是在「让模型朝奖励高的方向微调」和「别偏离太远(KL)」之间找平衡。

- 精简复习版见 [`./rl-unified.ipynb`](./rl-unified.ipynb)
- 本章习题与解答见 [`./exercise-solutions.ipynb`](./exercise-solutions.ipynb)

下一章:[第 14 章 · 智能体 RL](../ch14/01_main-chapter-code/README.md)